<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-04-image-and-multimodal-generation/lesson-4.2-stable-diffusion-ecosystem/notebooks/exercises-4.2.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 4.2 — Stable Diffusion Ecosystem
Netsetos GenAI Course v2.0 | Module 4

Covers: txt2img, img2img, ControlNet, LoRA, ComfyUI, Flux, optimization, deployment


In [ ]:
# Setup (GPU required for actual generation)
# pip install diffusers transformers torch accelerate controlnet-aux -q
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem/1e9:.1f}GB')


## Ex 1: Text-to-Image with SDXL


In [ ]:
# from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler
# pipe = StableDiffusionXLPipeline.from_pretrained(
#     'stabilityai/stable-diffusion-xl-base-1.0',
#     torch_dtype=torch.float16, variant='fp16'
# ).to('cuda')
# pipe.scheduler = DPMSolverMultistepScheduler.from_config(
#     pipe.scheduler.config, algorithm_type='dpmsolver++', use_karras_sigmas=True
# )
# image = pipe(
#     prompt='Indian wedding reception, elegant mandap, fairy lights, golden hour',
#     negative_prompt='blurry, distorted, low quality',
#     num_inference_steps=25, guidance_scale=7.5,
#     width=1024, height=1024,
#     generator=torch.Generator('cuda').manual_seed(42)
# ).images[0]
# image.save('wedding.png')
print('Uncomment and run on GPU. SDXL needs ~7GB VRAM.')


## Ex 2: Image-to-Image


In [ ]:
# from diffusers import StableDiffusionXLImg2ImgPipeline
# from PIL import Image
# pipe_i2i = StableDiffusionXLImg2ImgPipeline.from_pretrained(
#     'stabilityai/stable-diffusion-xl-base-1.0',
#     torch_dtype=torch.float16, variant='fp16'
# ).to('cuda')
# init_image = Image.open('input_photo.jpg').resize((1024, 1024))
# result = pipe_i2i(
#     prompt='Professional product photo, studio lighting, white background',
#     image=init_image, strength=0.65,  # 0=keep original, 1=ignore
#     num_inference_steps=25, guidance_scale=7.5
# ).images[0]
# result.save('product_photo.png')
print('img2img transforms existing images. strength controls how much to change.')


## Ex 3: ControlNet — Pose-Guided Generation


In [ ]:
# from diffusers import ControlNetModel, StableDiffusionXLControlNetPipeline
# from controlnet_aux import OpenposeDetector
# controlnet = ControlNetModel.from_pretrained(
#     'thibaud/controlnet-openpose-sdxl-1.0', torch_dtype=torch.float16
# )
# pipe_cn = StableDiffusionXLControlNetPipeline.from_pretrained(
#     'stabilityai/stable-diffusion-xl-base-1.0',
#     controlnet=controlnet, torch_dtype=torch.float16
# ).to('cuda')
# pose = OpenposeDetector.from_pretrained('lllyasviel/Annotators')
# pose_image = pose(Image.open('model_photo.jpg'))
# result = pipe_cn(
#     prompt='Indian woman in traditional Banarasi saree, studio lighting',
#     image=pose_image, controlnet_conditioning_scale=0.8
# ).images[0]
print('ControlNet: input pose skeleton → output follows the same pose')


## Ex 4: LoRA — Brand-Specific Style


In [ ]:
# Load a pre-trained LoRA
# pipe.load_lora_weights('path/to/jewelry_style_lora')
# pipe.fuse_lora(lora_scale=0.8)  # Merge for faster inference
# image = pipe('Gold necklace, Indian traditional design').images[0]
# pipe.unfuse_lora()  # Remove to switch styles

# Training your own LoRA (CLI)
# autotrain dreambooth \
#   --model stabilityai/stable-diffusion-xl-base-1.0 \
#   --data-path ./my_brand_images/ \
#   --prompt 'photo of sks brand style' \
#   --resolution 1024 --batch-size 1 --num-steps 1000 \
#   --lr 1e-4 --lora-rank 32
print('LoRA: 4-150MB adapter. Train in 30 min on 1 GPU. 20-50 images needed.')


## Ex 5: Prompt Engineering Techniques


In [ ]:
# Prompt weighting (requires compel library)
# from compel import Compel
# compel = Compel(tokenizer=pipe.tokenizer, text_encoder=pipe.text_encoder)
# prompt_embeds = compel('(golden:1.5) temple, (sunset:1.3) sky, (detailed:0.8) architecture')

# CLIP Skip (for anime/illustration models)
# image = pipe(prompt='anime girl', clip_skip=2).images[0]

# Negative prompt best practices
neg = 'blurry, distorted, extra limbs, bad anatomy, low quality, watermark, text'
print(f'Standard negative prompt: {neg}')
print('\nPrompt weighting: (word:1.5) = 1.5× emphasis')
print('CLIP skip 2: better for anime models trained at that setting')


## Ex 6: Scheduler Comparison


In [ ]:
from diffusers import (
    DDPMScheduler, DDIMScheduler, EulerDiscreteScheduler,
    DPMSolverMultistepScheduler, UniPCMultistepScheduler
)

schedulers = {
    'DDPM':     (DDPMScheduler, 1000, 'Reference quality, very slow'),
    'DDIM':     (DDIMScheduler, 50, 'Deterministic, good quality'),
    'Euler':    (EulerDiscreteScheduler, 25, 'Fast, default for many UIs'),
    'DPM++ 2M': (DPMSolverMultistepScheduler, 20, 'RECOMMENDED — best tradeoff'),
    'UniPC':    (UniPCMultistepScheduler, 15, 'Fastest, slight quality loss'),
}

print(f'{"Scheduler":12s} {"Steps":>6s}  {"Notes"}')
print('-' * 55)
for name, (cls, steps, notes) in schedulers.items():
    rec = ' ← USE THIS' if name == 'DPM++ 2M' else ''
    print(f'{name:12s} {steps:>6d}  {notes}{rec}')


## Ex 7: Cost Calculator


In [ ]:
def cost_comparison(images_per_month):
    platforms = {
        'DALL-E 3 API':  images_per_month * 0.04,
        'Replicate Flux': images_per_month * 0.003,
        'fal.ai Schnell':images_per_month * 0.003,
        'Fireworks SDXL': images_per_month * 0.004,
        'Modal A10G':    images_per_month * 0.0012,
        'RunPod A100':   images_per_month * 0.0024,
        'Jarvislabs L4': images_per_month * 0.004,  # ₹0.30 ≈ $0.004
    }
    print(f'\nCost for {images_per_month:,} images/month:')
    print(f'{"Platform":20s} {"USD":>8s} {"INR":>10s}')
    print('-' * 42)
    for name, cost in sorted(platforms.items(), key=lambda x: x[1]):
        print(f'{name:20s} ${cost:>7.2f} ₹{cost*84:>9,.0f}')

cost_comparison(10000)
cost_comparison(100000)


## Ex 8: Safety Checklist


In [ ]:
safety_checklist = [
    ('NSFW Safety Checker', 'Built into diffusers. NEVER disable in production.', True),
    ('Prompt Blocklist', 'Filter explicit/harmful keywords before generation.', True),
    ('Post-gen Moderation', 'AWS Rekognition ($1/1K) or Azure Content Safety.', True),
    ('C2PA Watermark', 'Cryptographic provenance metadata. ISO standard.', True),
    ('SynthID', 'Pixel-level watermark. Survives crops/resizes.', False),
    ('IT Rules 2026 Label', '10% area label for AI content. 3-hour takedown.', True),
    ('Celebrity Detection', 'RetinaFace + ArcFace against protected DB.', False),
    ('Indic Text Workaround', 'Generate without text → PIL compositing.', True),
]

print('Production Safety Checklist:')
for name, desc, required in safety_checklist:
    status = '🔴 REQUIRED' if required else '🟡 Recommended'
    print(f'  {status}: {name}')
    print(f'           {desc}')


---
## Recap
The SD ecosystem in 2026: SDXL for ecosystem depth, Flux for quality. ComfyUI for workflows, diffusers for code. DPM++ 2M Karras scheduler, 25 steps. Post-processing: ADetailer→CodeFormer→Real-ESRGAN. Deploy on Modal/RunPod for cost efficiency. Indian GPU clouds save 50-70%. Comply with IT Amendment Rules 2026.
